In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

print("Libraries imported successfully!")


: 

In [ ]:

df = pd.read_csv("processed_DrugSolubilityData.csv")

df.head()

: 

In [ ]:

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


In [ ]:
df.info()

In [ ]:
df[["value", "Temperature, K"]].describe().round(3)

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="value",
    bins=40,
    kde=True,
    color="steelblue"
)

plt.title("Distribution of Log Solubility")
plt.xlabel("Reported Log Solubility")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

sns.histplot(data=df, x="value", bins=40, kde=True,
             color="steelblue")

plt.title("Distribution of Log Solubility")
plt.xlabel("Reported Log Solubility")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.savefig("solubility_distribution.png", dpi=300)
plt.show()


Figure 1. Distribution of Reported Log Solubility

The histogram shows a negatively skewed distribution of reported log-solubility values, with most observations concentrated between approximately -1 and 1.5 and a peak near 0.8. A long left tail extends toward substantially lower solubility values. This indicates an uneven distribution of the regression target, which may affect model performance, particularly for compounds with low solubility. Therefore, prediction errors should be examined across different solubility ranges.

In [ ]:

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="Temperature, K",
    bins=40,
    color="steelblue"
)

plt.title("Distribution of Experimental Temperature")
plt.xlabel("Temperature (K)")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.savefig("temperature_distribution.png", dpi=300)
plt.show()


Figure 2. Distribution of Experimental Temperature

The temperature distribution shows that most solubility measurements were collected between approximately 275 K and 325 K, with the highest frequencies around 295-315 K. The histogram contains several distinct peaks, suggesting that measurements were performed at selected experimental temperatures rather than uniformly across the entire range. Relatively few observations occur at extreme temperatures. This uneven coverage may affect the generalization of machine learning models, particularly when predicting solubility under less frequently represented temperature conditions.

In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Temperature, K",
    y="value",
    alpha=0.25,
    s=12
)

plt.title("Temperature vs. Log Solubility")
plt.xlabel("Temperature (K)")
plt.ylabel("Reported Log Solubility")

plt.tight_layout()
plt.savefig("temperature_vs_solubility.png", dpi=300)
plt.show()


Figure 3. Relationship Between Temperature and Log Solubility

The scatterplot illustrates substantial variation in reported log solubility at similar temperatures, with no clear overall linear relationship. Vertical clusters indicate that measurements were collected at discrete experimental temperatures. The considerable variation in solubility at a fixed temperature suggests that additional variables, particularly chemical identity and solvent composition, are important for predicting solubility. Consequently, temperature alone is unlikely to provide sufficient predictive information across the heterogeneous dataset.

In [ ]:

correlation = df[
    ["Temperature, K", "value"]
].corr(method="pearson")

print("Pearson Correlation Matrix:")
print(correlation.round(3))

plt.figure(figsize=(6, 4))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".3f"
)

plt.title("Correlation Between Temperature and Log Solubility")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300)
plt.show()


Figure 4. Pearson Correlation Between Temperature and Log Solubility

Pearson correlation analysis revealed a weak positive linear association between experimental temperature and reported log solubility (r = 0.179). The corresponding coefficient of determination (r² ≈ 0.032) indicates that temperature alone accounts for approximately 3.2% of the variation in log solubility under a simple linear regression model. This finding suggests that temperature is insufficient as a standalone predictor across the heterogeneous dataset and supports incorporating molecular identity and solvent composition into subsequent predictive models.

In [ ]:

plt.figure(figsize=(9, 4))

sns.boxplot(
    data=df,
    x="value",
    color="steelblue"
)

plt.title("Boxplot of Reported Log Solubility")
plt.xlabel("Reported Log Solubility")

plt.tight_layout()
plt.savefig("solubility_boxplot.png", dpi=300)
plt.show()


Figure 5. Boxplot of Reported Log Solubility

The boxplot reveals a negatively skewed distribution with numerous potential outliers at low log-solubility values. The median is approximately 0.35, while the middle 50% of observations fall between approximately −0.4 and 1.0. Several observations extend below the lower whisker, reaching approximately −5.5. These extreme values may represent genuinely poorly soluble compounds rather than measurement errors. Therefore, they should be investigated before considering exclusion from subsequent machine learning analysis.

In [ ]:

Q1 = df["value"].quantile(0.25)
Q3 = df["value"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["value"] < lower_bound) |
    (df["value"] > upper_bound)
]

print("Q1:", round(Q1, 3))
print("Q3:", round(Q3, 3))
print("IQR:", round(IQR, 3))
print("Lower bound:", round(lower_bound, 3))
print("Upper bound:", round(upper_bound, 3))
print("Number of potential outliers:", len(outliers))
print("Percentage of potential outliers:",
      round(len(outliers) / len(df) * 100, 2), "%")


Outlier Analysis

The interquartile range (IQR) method identified 271 potential outliers, representing 1.0% of the dataset. The calculated lower and upper thresholds were −2.567 and 3.162, respectively. The extreme observations are concentrated toward the low-solubility region, consistent with the negatively skewed target distribution. These observations should not be automatically excluded, as they may represent chemically meaningful measurements. Their validity and potential influence on model performance will be investigated during subsequent analysis.

In [ ]:

print("Dataset columns:")
for column in df.columns:
    print(column)

print("\nFirst five observations:")
display(df.head())


In [ ]:

print("Unique drug identifier entries:",
      df["cmp_ids_drug"].nunique())

print("Unique solvent identifier entries:",
      df["cmp_ids_solvent"].nunique())

print("\nMost frequent drug entries:")
print(df["cmp_ids_drug"].value_counts().head(10))

print("\nMost frequent solvent entries:")
print(df["cmp_ids_solvent"].value_counts().head(10))


In [ ]:

top_solvents = df["cmp_ids_solvent"].value_counts().head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=top_solvents.values,
    y=top_solvents.index.astype(str),
    color="steelblue"
)

plt.title("Top 10 Most Represented Solvent Systems")
plt.xlabel("Number of Observations")
plt.ylabel("Solvent Identifier Pair")

plt.tight_layout()
plt.savefig("top_10_solvent_systems.png", dpi=300)
plt.show()


Chemical and Solvent-System Representation

The dataset contains 125 distinct drug identifier entries and 166 distinct solvent identifier entries. The distribution of observations across solvent systems is uneven, with the most frequently represented solvent pair, [130.0, 129.0], contributing 3,009 observations(approximately 
11.1% of the dataset). This imbalance suggests that some chemical environments are substantially better represented than others. Consequently, model evaluation should consider performance across different solvent systems rather than relying exclusively on aggregate prediction metrics.

Figure 6. Distribution of Solvent-System Observations

The distribution of observations across solvent systems is highly uneven. The most frequently represented solvent identifier pair, [130.0, 129.0], accounts for 3,009 observations, approximately 11.1% of the dataset. Several other solvent systems also contribute substantial numbers of measurements, whereas many systems are less frequently represented. Additionally, some solvent identifier pairs occur in reversed order, indicating a need to examine how solvent components and their associated compositions are encoded. This imbalance may influence model training and evaluation, particularly for underrepresented solvent systems.

In [ ]:

print("All dataset columns:")
print(df.columns.tolist())

print("\nFirst three complete observations:")
for i in range(3):
    print(f"\nObservation {i + 1}")
    print(df.iloc[i].to_string())


In [ ]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
import ast
import numpy as np

def parse_fractions(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

solvent_fractions = df["cmp_mole_fractions_solvent"].apply(parse_fractions)

fraction_sums = solvent_fractions.apply(
    lambda x: sum(x)
)

print("Solvent mole fraction statistics:")
print(fraction_sums.describe())

print("\nMixtures not summing to approximately 1:")
print((~np.isclose(fraction_sums, 1.0, atol=1e-4)).sum())

print("\nNumber of solvents per observation:")
print(solvent_fractions.apply(len).value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt

solvent_counts = solvent_fractions.apply(len).value_counts().sort_index()

plt.figure(figsize=(7, 5))

plt.bar(
    ["Single solvent", "Two solvents"],
    [solvent_counts.get(1, 0), solvent_counts.get(2, 0)]
)

plt.title("Distribution of Solvent Mixture Types")
plt.xlabel("Solvent System")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.savefig("solvent_mixture_types.png", dpi=300)
plt.show()

In [ ]:
df["solvent_type"] = solvent_fractions.apply(
    lambda x: "Single solvent" if len(x) == 1 else "Two solvents"
)

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="solvent_type",
    y="value"
)

plt.title("Log Solubility by Solvent System Type")
plt.xlabel("Solvent System")
plt.ylabel("Reported Log Solubility")

plt.tight_layout()
plt.savefig("solubility_by_solvent_type.png", dpi=300)
plt.show()

print("Solubility statistics by solvent type:")
print(df.groupby("solvent_type")["value"].describe())

Solubility Distribution by Solvent System

The dataset contains 2,943 single-solvent observations and 24,223 binary-solvent observations. Binary-solvent systems exhibit a slightly higher mean reported log solubility (0.240) compared with single-solvent systems (0.092). The median values are relatively similar (0.380 and 0.325, respectively). Single-solvent systems show greater variability, with a standard deviation of 1.242 compared with 1.017 for binary-solvent systems. Both groups contain low-solubility outliers. These findings suggest that solvent-system characteristics may be relevant predictors of drug solubility. However, differences between the groups cannot be attributed solely to the number of solvents because drug identity, solvent identity, composition, and temperature may also influence the measured values.

In [ ]:
binary_df = df[df["solvent_type"] == "Two solvents"].copy()

binary_df["solvent_1_fraction"] = (
    solvent_fractions.loc[binary_df.index].apply(lambda x: x[0])
)

plt.figure(figsize=(9, 5))

sns.scatterplot(
    data=binary_df,
    x="solvent_1_fraction",
    y="value",
    alpha=0.2,
    s=10
)

plt.title("Solvent Mole Fraction vs. Log Solubility")
plt.xlabel("Mole Fraction of First Listed Solvent")
plt.ylabel("Reported Log Solubility")

plt.tight_layout()
plt.savefig("solvent_fraction_vs_solubility.png", dpi=300)
plt.show()

Effect of Solvent Composition on Log Solubility.

The scatter plot illustrates the relationship between the mole fraction of the first listed solvent and reported log solubility for binary-solvent systems. Solubility values vary substantially across the entire composition range, with no clear overall linear trend. This variability may reflect differences in drug identity, solvent combinations, and experimental temperature. Consequently, solvent composition should be evaluated alongside other chemical and experimental variables when developing predictive machine learning models.

In [ ]:
correlation = binary_df[
    ["solvent_1_fraction", "value"]
].corr()

print("Correlation between solvent mole fraction and log solubility:")
print(correlation)

Correlation Analysis: Solvent Composition and Log Solubility.

Pearson correlation analysis revealed a positive correlation (r = 0.3544) between the mole fraction of the first listed solvent and reported log solubility in binary-solvent systems. This indicates a weak-to-moderate overall linear association. However, the dataset contains multiple drug identities, solvent combinations, and experimental temperatures. Consequently, the observed relationship may reflect differences between chemical systems rather than a consistent effect of solvent composition within individual mixtures.

In [ ]:
# Select the most frequently represented
# drug-solvent combination

system_counts = (
    binary_df.groupby(
        ["cmp_ids_drug", "cmp_ids_solvent"]
    )
    .size()
    .sort_values(ascending=False)
)

print("Top 10 drug-solvent combinations:")
print(system_counts.head(10))

In [ ]:

# Select one drug-solvent combination
selected_drug = "[113]"
selected_solvent = "[144.0, 129.0]"

# Filter the dataset
selected_df = binary_df[
    (binary_df["cmp_ids_drug"].astype(str) == selected_drug) &
    (binary_df["cmp_ids_solvent"].astype(str) == selected_solvent)
].copy()

# Check the selected data
print("Number of observations:", len(selected_df))
print("Temperatures:", sorted(selected_df["Temperature, K"].unique()))
print(selected_df.head())


In [ ]:

# Plot solubility against solvent composition
# for one drug-solvent system

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=selected_df,
    x="solvent_1_fraction",
    y="value",
    hue="Temperature, K",
    palette="viridis",
    s=65
)

plt.title(
    "Effect of Solvent Composition and Temperature on Log Solubility"
)
plt.xlabel("Mole Fraction of First Solvent")
plt.ylabel("Reported Log Solubility")
plt.legend(title="Temperature (K)", bbox_to_anchor=(1.05, 1))
plt.tight_layout()

plt.savefig("specific_system_solubility.png", dpi=300)
plt.show()


Effect of Solvent Composition and Temperature on Drug Solubility

To investigate the influence of experimental conditions on drug solubility, a representative drug–solvent system was selected for detailed analysis. The selected system consisted of drug identifier [113] and solvent identifiers [144.0, 129.0], containing 135 experimental observations. The scatter plot demonstrated that reported log solubility generally increased with both temperature and the mole fraction of the first solvent. At constant solvent composition, higher temperatures were associated with higher solubility values. Similarly, at comparable temperatures, increasing the proportion of the first solvent resulted in higher solubility, although the relationship appeared nonlinear.
These observations suggest that both temperature and solvent composition are relevant predictors of drug solubility. The results also highlight the importance of considering chemical identity and experimental conditions when developing machine learning models for solubility prediction.

In [ ]:

# Correlation analysis for selected drug-solvent system

correlation = selected_df[
    ["Temperature, K", "solvent_1_fraction", "value"]
].corr()

print("Correlation matrix:")
print(correlation.round(3))

plt.figure(figsize=(7, 5))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".3f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Analysis for Selected Drug-Solvent System")
plt.tight_layout()
plt.savefig("specific_system_correlation.png", dpi=300)
plt.show()


Correlation Analysis of a Representative Drug–Solvent System

Pearson correlation analysis was performed on 135 experimental observations corresponding to drug identifier [113] and solvent system [144.0, 129.0]. A strong positive correlation (r = 0.877) was observed between experimental temperature and reported log solubility, indicating that higher temperatures were associated with increased solubility within this system. The mole fraction of the first solvent exhibited a weaker positive correlation with log solubility (r = 0.355). This suggests that solvent composition also contributes to solubility variation, although the relationship may be nonlinear. The correlation between temperature and solvent mole fraction was approximately zero, indicating no substantial linear association between these two experimental variables in the selected observations. Overall, the results support the inclusion of temperature and solvent composition as input features in the proposed machine learning model. Drug and solvent identities should also be considered to account for differences in chemical systems.

In [ ]:
# inspecting the processed data

import numpy as np
import pandas as pd

df = pd.read_csv(r"C:\Users\cheny\Desktop\CHE1147 group project\chemixhub\datasets\drug-solubility\processed_data\processed_DrugSolubilityData.csv")

print(df.head())
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Column names:", df.columns.tolist())

In [ ]:
df.info()

print(df[["value", "Temperature, K"]].describe())

print(df[["property", "unit"]].value_counts(dropna=False))

In [ ]:
# examine missing values and duplications

print("Missing values per column:")
print(df.isna().sum())

print("Exactly duplicated rows:")
print(df.duplicated().sum())

In [ ]:
# inspect whether each row contains valid information about the drug and solvent mixture

import ast

# 1. Convert the original CSV text into Python lists
solvent_ids = df["cmp_ids_solvent"].apply(ast.literal_eval)

solvent_fractions = df["cmp_mole_fractions_solvent"].apply(
    ast.literal_eval
)

# 2. Count the items in each list
id_counts = solvent_ids.apply(len)
fraction_counts = solvent_fractions.apply(len)

# 3. Compare the counts
mismatch = id_counts != fraction_counts

# 4. Count and display affected rows
print(
    "Rows with different list lengths:",
    np.count_nonzero(mismatch)
)

print(
    df.loc[
        mismatch,
        ["cmp_ids_solvent", "cmp_mole_fractions_solvent"]
    ].head(10)
)


In [ ]:
# find the correct pairing

raw = pd.read_csv(r"C:\Users\cheny\Desktop\CHE1147 group project\chemixhub\datasets\drug-solubility\raw_data\DrugSolubilityData.csv")

print("Raw rows:", len(raw))
print("Processed rows:", len(df))

assert len(raw) == len(df), "Row counts differ"

assert (
    raw["LogS"].to_numpy() == df["value"].to_numpy()
).all(), "Measurement values differ"

assert (
    raw["Temperature, K"].to_numpy()
    == df["Temperature, K"].to_numpy()
).all(), "Temperatures differ"

print("Initial alignment checks passed")

In [ ]:
print("Processed record:")
print(
    df.iloc[3815][
        ["cmp_ids_solvent", "cmp_mole_fractions_solvent"]
    ]
)

print("\nRaw record:")
print(
    raw.iloc[3815][
        ["SMILES_1", "comp_1", "SMILES_2", "comp_2"]
    ]
)

In [ ]:
compounds = pd.read_csv(r"C:\Users\cheny\Desktop\CHE1147 group project\chemixhub\datasets\drug-solubility\processed_data\compounds.csv")

print(compounds[["compound_id", "smiles"]].head())

solvent_matches = compounds["smiles"].str.strip().isin(["OC", "O"])

print(
    compounds.loc[
        solvent_matches,
        ["compound_id", "smiles"]
    ]
)

affected_compositions = raw.loc[
    mismatch,
    ["comp_1", "comp_2"]
]

print("\nAffected compositions:")
print(affected_compositions.value_counts(dropna=False))


In [ ]:
assert raw.index.equals(df.index), "Raw and processed indices differ"

# Identify pure-solvent observations in the processed dataset
processed_fractions = df[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

pure_solvent_mask = processed_fractions.apply(
    lambda fractions: (
        sum(fraction > 0 for fraction in fractions) == 1
        and np.isclose(sum(fractions), 1.0, rtol=0, atol=1e-6)
    )
)

# Check for missing or blank identifiers in either solvent column
def is_missing_identifier(column):
    return (
        column.isna()
        | column.astype("string").str.strip().eq("").fillna(False)
    )

missing_first = is_missing_identifier(raw["SMILES_1"])
missing_second = is_missing_identifier(raw["SMILES_2"])

# Keep pure-solvent rows with at least one missing identifier
matching_mask = (
    pure_solvent_mask
    & (missing_first | missing_second)
)

raw_pure_missing_solvent = raw.loc[
    matching_mask,
    [
        "SMILES_0",
        "SMILES_1", "comp_1",
        "SMILES_2", "comp_2",
        "Temperature, K", "LogS"
    ]
].copy()

# Show which identifier is missing
raw_pure_missing_solvent["missing_SMILES_1"] = missing_first.loc[
    matching_mask
]

raw_pure_missing_solvent["missing_SMILES_2"] = missing_second.loc[
    matching_mask
]

# Include the processed representation for comparison
raw_pure_missing_solvent["processed_solvent_ids"] = df.loc[
    matching_mask, "cmp_ids_solvent"
]

raw_pure_missing_solvent["processed_solvent_fractions"] = df.loc[
    matching_mask, "cmp_mole_fractions_solvent"
]

raw_pure_missing_solvent.index.name = "original_row_index"

# Report mutually exclusive counts
print("Total pure-solvent rows:", int(pure_solvent_mask.sum()))

print(
    "Missing only SMILES_1:",
    int((pure_solvent_mask & missing_first & ~missing_second).sum())
)

print(
    "Missing only SMILES_2:",
    int((pure_solvent_mask & ~missing_first & missing_second).sum())
)

print(
    "Missing both identifiers:",
    int((pure_solvent_mask & missing_first & missing_second).sum())
)

print("Total matching rows:", len(raw_pure_missing_solvent))

display(raw_pure_missing_solvent)

In [ ]:
smiles = compounds["smiles"].str.strip()

assert not smiles.duplicated().any()
assert not compounds["compound_id"].duplicated().any()

smiles_to_id = dict(zip(smiles, compounds["compound_id"]))

solvent_ids = df["cmp_ids_solvent"].apply(ast.literal_eval)

solvent_fractions = df[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

# Identify length mismatches and existing single-solvent rows
mismatch = (
    solvent_ids.apply(len) != solvent_fractions.apply(len)
)

single_solvent_mask = (
    solvent_ids.apply(len).eq(1)
    & solvent_fractions.apply(len).eq(1)
)

repair_mask = mismatch | single_solvent_mask

def get_raw_solvent_pair(idx):
    """Recover both solvent slots in their original raw-data order."""
    ids = []
    fractions = []

    for position in (1, 2):

        raw_smiles = raw.at[idx, f"SMILES_{position}"]
        raw_percent = raw.at[idx, f"comp_{position}"]

        missing_identifier = (
            pd.isna(raw_smiles)
            or (
                isinstance(raw_smiles, str)
                and not raw_smiles.strip()
            )
        )

        if missing_identifier:

            # Only an absent, zero-fraction solvent may receive "N/A".
            assert pd.isna(raw_percent) or float(raw_percent) == 0.0, (
                f"Missing solvent identifier with nonzero composition "
                f"at row {idx}, position {position}"
            )

            ids.append("N/A")
            fractions.append(0.0)

        else:

            assert isinstance(raw_smiles, str), (
                f"Unexpected solvent identifier at row {idx}"
            )

            assert pd.notna(raw_percent), (
                f"Missing composition for a named solvent at row {idx}"
            )

            solvent_smiles = raw_smiles.strip()

            assert solvent_smiles in smiles_to_id, (
                f"Unknown solvent SMILES at row {idx}: {solvent_smiles}"
            )

            ids.append(int(smiles_to_id[solvent_smiles]))
            fractions.append(float(raw_percent) / 100.0)

    return ids, fractions


repaired = df.copy()

for idx in df.index[repair_mask]:

    raw_ids, fractions = get_raw_solvent_pair(idx)

    assert fractions in ([1.0, 0.0], [0.0, 1.0]), (
        f"Unexpected raw composition at row {idx}: {fractions}"
    )

    assert solvent_fractions.at[idx] == [1.0], (
        f"Unexpected original fraction list at row {idx}"
    )

    if mismatch.at[idx]:

        assert solvent_ids.at[idx] == raw_ids, (
            f"Solvent IDs disagree with raw data at row {idx}"
        )

    else:

        active_id = raw_ids[fractions.index(1.0)]

        assert solvent_ids.at[idx] == [active_id], (
            f"Pure-solvent ID disagrees with raw data at row {idx}"
        )

    # Save both slots in the raw-data order.
    repaired.at[idx, "cmp_ids_solvent"] = str(raw_ids)

    repaired.at[
        idx, "cmp_mole_fractions_solvent"
    ] = str(fractions)


print("Length mismatches repaired:", int(mismatch.sum()))
print("Single-solvent rows expanded:", int(single_solvent_mask.sum()))
print("Total rows updated:", int(repair_mask.sum()))
print("Rows retained:", len(repaired))

In [ ]:
# Verify the repaired solvent IDs and fractions
repaired_ids = repaired["cmp_ids_solvent"].apply(ast.literal_eval)

repaired_fractions = repaired[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

# Check if all rows contain two solvent slots and two fractions
assert repaired_ids.apply(len).eq(2).all(), (
    "Some rows do not contain two solvent slots"
)

assert repaired_fractions.apply(len).eq(2).all(), (
    "Some rows do not contain two solvent fractions"
)

In [ ]:
# Validate IDs, fractions, and N/A positions
known_ids = set(compounds["compound_id"])

for idx in repaired.index:

    ids = repaired_ids.at[idx]
    fractions = repaired_fractions.at[idx]

    assert all(
        isinstance(f, (int, float))
        and not isinstance(f, bool)
        and np.isfinite(f)
        and 0 <= f <= 1
        for f in fractions
    ), f"Invalid fractions at row {idx}"

    assert np.isclose(
        sum(fractions), 1.0, rtol=0, atol=1e-6
    ), f"Fractions do not sum to 1 at row {idx}"

    actual_ids = [cid for cid in ids if cid != "N/A"]

    assert len(actual_ids) == len(set(actual_ids)), (
        f"Repeated solvent ID at row {idx}"
    )

    for cid, fraction in zip(ids, fractions):

        if cid == "N/A":
            assert fraction == 0.0, (
                f"N/A has a nonzero fraction at row {idx}"
            )

        else:
            assert (
                isinstance(cid, (int, float))
                and not isinstance(cid, bool)
                and np.isfinite(cid)
                and cid == int(cid)
                and cid in known_ids
            ), f"Invalid solvent ID at row {idx}: {cid}"

In [ ]:
# verify every updated row against the raw data
for idx in repaired.index[repair_mask]:

    ids = repaired_ids.at[idx]
    fractions = repaired_fractions.at[idx]

    for slot, position in enumerate((1, 2)):

        raw_smiles = raw.at[idx, f"SMILES_{position}"]
        raw_percent = raw.at[idx, f"comp_{position}"]

        missing_identifier = (
            pd.isna(raw_smiles)
            or (
                isinstance(raw_smiles, str)
                and not raw_smiles.strip()
            )
        )

        if missing_identifier:

            assert ids[slot] == "N/A", (
                f"Expected N/A at row {idx}, solvent {position}"
            )

            assert fractions[slot] == 0.0, (
                f"Expected zero fraction at row {idx}, solvent {position}"
            )

            assert pd.isna(raw_percent) or float(raw_percent) == 0.0, (
                f"Raw composition contradicts N/A at row {idx}"
            )

        else:

            expected_id = int(smiles_to_id[raw_smiles.strip()])

            assert ids[slot] == expected_id, (
                f"ID or position disagrees with raw data at row {idx}, "
                f"solvent {position}"
            )

            assert pd.notna(raw_percent), (
                f"Missing raw composition at row {idx}, solvent {position}"
            )

            assert np.isclose(
                fractions[slot],
                float(raw_percent) / 100.0,
                rtol=0,
                atol=1e-12
            ), (
                f"Fraction disagrees with raw data at row {idx}, "
                f"solvent {position}"
            )

In [ ]:
# verify that only the intended rows and columns changed

assert repaired.index.equals(df.index)

original_ids = df["cmp_ids_solvent"].apply(ast.literal_eval)
updated_ids = repaired["cmp_ids_solvent"].apply(ast.literal_eval)

original_fractions = df[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

updated_fractions = repaired[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

# Numeric IDs such as 127 and 127.0 compare as equal
changed_ids = pd.Series(
    [
        before != after
        for before, after in zip(original_ids, updated_ids)
    ],
    index=df.index
)

changed_fractions = pd.Series(
    [
        before != after
        for before, after in zip(original_fractions, updated_fractions)
    ],
    index=df.index
)

# Only the original single-solvent rows should gain an ID slot

assert changed_ids.eq(single_solvent_mask).all(), (
    "Unexpected changes to solvent ID contents"
)

# Both mismatched and single-solvent rows should gain a fraction slot

assert changed_fractions.eq(repair_mask).all(), (
    "Unexpected changes to solvent fraction contents"
)

print("Rows with changed ID contents:", int(changed_ids.sum()))
print("Rows with changed fraction contents:", int(changed_fractions.sum()))
solvent_columns = [
    "cmp_ids_solvent",
    "cmp_mole_fractions_solvent"
]

assert repaired.drop(columns=solvent_columns).equals(
    df.drop(columns=solvent_columns)
), "An unrelated column changed"

assert repaired.loc[~repair_mask].equals(
    df.loc[~repair_mask]
), "A row outside the repair mask changed"

# confirm no rows were removed or reordered

assert len(repaired) == len(df)
assert repaired.index.equals(df.index)

# report the verified results

missing_first_count = updated_ids.apply(
    lambda ids: ids[0] == "N/A"
).sum()

missing_second_count = updated_ids.apply(
    lambda ids: ids[1] == "N/A"
).sum()

print("Length mismatches repaired:", int(mismatch.sum()))
print("Single-solvent rows expanded:", int(single_solvent_mask.sum()))
print("N/A in first solvent slot:", int(missing_first_count))
print("N/A in second solvent slot:", int(missing_second_count))
print("Rows retained:", len(repaired))
print("All repair verification checks passed.")

In [ ]:
# check the duplication after repair

def active_composition(ids_text, fractions_text):
    ids = ast.literal_eval(ids_text)
    fractions = ast.literal_eval(fractions_text)

    assert len(ids) == len(fractions)

    # Ignore zero-fraction slots before converting IDs to integers.
    # This also excludes "N/A", whose fraction was verified as zero.
    return tuple(sorted(
        (int(cid), float(fraction))
        for cid, fraction in zip(ids, fractions)
        if fraction > 0
    ))


# Preserve all other fields in the duplicate comparison,

composition_columns = [
    "cmp_ids_drug",
    "cmp_mole_fractions_drug",
    "cmp_ids_solvent",
    "cmp_mole_fractions_solvent"
]

duplicate_keys = repaired.drop(columns=composition_columns).copy()

for kind in ["drug", "solvent"]:
    duplicate_keys[f"active_{kind}_composition"] = [
        active_composition(ids, fractions)
        for ids, fractions in zip(
            repaired[f"cmp_ids_{kind}"],
            repaired[f"cmp_mole_fractions_{kind}"]
        )
    ]

duplicate_mask = duplicate_keys.duplicated(keep="first")

duplicate_rows = repaired.loc[
    duplicate_keys.duplicated(keep=False)
]

print("Exact text-based duplicates:", int(repaired.duplicated().sum()))
print("Duplicate rows to remove:", int(duplicate_mask.sum()))

display(duplicate_rows)

In [ ]:
# for duplications, keeping the first occurrence

removed_duplicates = repaired.loc[duplicate_mask].copy()
cleaned = repaired.loc[~duplicate_mask].copy()

# Verify removal using the same comparison keys
assert not duplicate_keys.loc[cleaned.index].duplicated().any()
assert len(cleaned) + len(removed_duplicates) == len(repaired)

print("Rows before duplicate removal:", len(repaired))
print("Duplicate rows removed:", len(removed_duplicates))
print("Rows after duplicate removal:", len(cleaned))

In [ ]:
# inspect whether each row contains valid information after cleaning

values = cleaned["value"]
temperatures = cleaned["Temperature, K"]

# Flag invalid measurements
invalid_value = ~np.isfinite(values)

# Flag invalid temperatures
invalid_temperature = (
    ~np.isfinite(temperatures)
    | (temperatures <= 0)
)

print("Invalid measurement values:", invalid_value.sum())
print("Invalid temperatures:", invalid_temperature.sum())

invalid_numeric = invalid_value | invalid_temperature

print(
    cleaned.loc[
        invalid_numeric,
        ["value", "Temperature, K"]
    ].head(10)
)

In [ ]:
#check if ids and solvent are valid

known_ids = set(compounds["compound_id"])


def valid_ids(items, allow_missing=False):
    if not isinstance(items, list) or len(items) == 0:
        return False

    actual_ids = []

    for item in items:

        if item == "N/A":
            if not allow_missing:
                return False
            continue

        if isinstance(item, bool) or not isinstance(item, (int, float)):
            return False

        if not np.isfinite(item):
            return False

        if item < 0 or item != int(item) or item not in known_ids:
            return False

        actual_ids.append(int(item))

    # Require at least one real compound and no repeated real IDs
    return (
        len(actual_ids) > 0
        and len(actual_ids) == len(set(actual_ids))
    )


def valid_fractions(items):
    if not isinstance(items, list) or len(items) == 0:
        return False

    for item in items:

        if isinstance(item, bool) or not isinstance(item, (int, float)):
            return False

        if not np.isfinite(item) or not 0 <= item <= 1:
            return False

    return np.isclose(
        sum(items),
        1.0,
        rtol=0,
        atol=1e-6
    )


for kind in ["drug", "solvent"]:

    ids = cleaned[f"cmp_ids_{kind}"].apply(ast.literal_eval)

    fractions = cleaned[
        f"cmp_mole_fractions_{kind}"
    ].apply(ast.literal_eval)

    # N/A is allowed only in solvent lists
    bad_ids = ~ids.apply(
        lambda items: valid_ids(
            items,
            allow_missing=(kind == "solvent")
        )
    )

    bad_fractions = ~fractions.apply(valid_fractions)

    bad_lengths = ids.apply(len) != fractions.apply(len)

    # This dataset should have one drug slot and two solvent slots
    expected_slots = 2 if kind == "solvent" else 1

    bad_slot_counts = (
        ids.apply(len).ne(expected_slots)
        | fractions.apply(len).ne(expected_slots)
    )

    # Any N/A must be a solvent slot paired with exactly zero
    bad_na_pairs = pd.Series(
        [
            any(
                cid == "N/A"
                and (kind != "solvent" or fraction != 0.0)
                for cid, fraction in zip(id_list, fraction_list)
            )
            for id_list, fraction_list in zip(ids, fractions)
        ],
        index=cleaned.index
    )

    print(f"\n{kind.upper()}")
    print("Invalid ID lists:", int(bad_ids.sum()))
    print("Invalid fraction lists:", int(bad_fractions.sum()))
    print("Length mismatches:", int(bad_lengths.sum()))
    print("Incorrect number of slots:", int(bad_slot_counts.sum()))
    print("Invalid N/A–fraction pairs:", int(bad_na_pairs.sum()))

    bad_rows = (
        bad_ids
        | bad_fractions
        | bad_lengths
        | bad_slot_counts
        | bad_na_pairs
    )

    if bad_rows.any():
        display(
            cleaned.loc[
                bad_rows,
                [f"cmp_ids_{kind}", f"cmp_mole_fractions_{kind}"]
            ].head(10)
        )

    assert not bad_rows.any(), (
        f"Review the invalid {kind} compositions shown above"
    )

print("\nAll ID and fraction validation checks passed.")

In [ ]:
# check if ID–fraction pairs in the cleaned data match the original raw data

def pairs_match(actual, expected):

    # Support both numeric IDs and the "N/A" placeholder.
    # Sort complete ID–fraction pairs so each fraction stays with its ID.
    sort_key = lambda pair: (
        pair[0] == "N/A",
        str(pair[0])
    )

    actual = sorted(actual, key=sort_key)
    expected = sorted(expected, key=sort_key)

    if len(actual) != len(expected):
        return False

    for (id_a, fraction_a), (id_b, fraction_b) in zip(actual, expected):

        if id_a != id_b:
            return False

        if not np.isclose(
            fraction_a,
            fraction_b,
            rtol=0,
            atol=1e-12
        ):
            return False

    return True


# Original indices must still identify the corresponding raw rows
assert cleaned.index.is_unique
assert cleaned.index.isin(raw.index).all()

pair_issues = []
numeric_issues = []

for idx, row in cleaned.iterrows():

    source = raw.loc[idx]

    # Verify measurement and temperature against the raw row
    for processed_column, raw_column in [
        ("value", "LogS"),
        ("Temperature, K", "Temperature, K")
    ]:
        if not np.isclose(
            row[processed_column],
            source[raw_column],
            rtol=0,
            atol=1e-12
        ):
            numeric_issues.append((idx, processed_column))

    # Verify drug and solvent compositions
    for kind in ["drug", "solvent"]:

        ids = ast.literal_eval(row[f"cmp_ids_{kind}"])

        fractions = ast.literal_eval(
            row[f"cmp_mole_fractions_{kind}"]
        )

        if len(ids) != len(fractions):
            pair_issues.append((idx, kind, "List length mismatch"))
            continue

        # Keep ALL entries, including zero fractions and N/A
        actual = [
            (
                "N/A" if cid == "N/A" else int(cid),
                float(fraction)
            )
            for cid, fraction in zip(ids, fractions)
        ]

        if kind == "solvent":

            # Use the helper defined in the repair cell
            expected_ids, expected_fractions = get_raw_solvent_pair(idx)

            expected = list(zip(
                expected_ids,
                expected_fractions
            ))

        else:

            drug_smiles = source["SMILES_0"].strip()

            expected = [
                (
                    int(smiles_to_id[drug_smiles]),
                    float(source["comp_0"]) / 100.0
                )
            ]

        if not pairs_match(actual, expected):
            pair_issues.append((idx, kind, "ID–fraction mismatch"))


print("Raw/cleaned composition mismatches:", len(pair_issues))
print("Raw/cleaned numeric mismatches:", len(numeric_issues))

print("First composition issues:", pair_issues[:10])
print("First numeric issues:", numeric_issues[:10])

assert not pair_issues, "Review the composition mismatches"
assert not numeric_issues, "Review the numeric mismatches"

print("All cleaned rows match their corresponding raw observations.")

In [ ]:
# Keep ALL drugs, including pure solvents and binary mixtures

subset = cleaned.copy()

solvent_fraction_lists = subset[
    "cmp_mole_fractions_solvent"
].apply(ast.literal_eval)

active_solvent_counts = solvent_fraction_lists.apply(
    lambda fractions: sum(fraction > 0 for fraction in fractions)
)

pure_solvent_mask = active_solvent_counts.eq(1)
binary_mask = active_solvent_counts.eq(2)

assert active_solvent_counts.isin([1, 2]).all()
assert len(subset) == len(cleaned)

print("Total observations:", len(subset))
print("Pure-solvent observations:", int(pure_solvent_mask.sum()))
print("Binary-mixture observations:", int(binary_mask.sum()))

drug_lists = subset["cmp_ids_drug"].apply(ast.literal_eval)

assert drug_lists.apply(len).eq(1).all(), (
    "Review observations containing more than one drug"
)

subset_drug_ids = drug_lists.apply(lambda ids: int(ids[0]))

drug_names = {
    int(cid): f"Drug {int(cid)}"
    for cid in subset_drug_ids.unique()
}

drug_names.update({
    5: "Itraconazole",
    43: "Posaconazole",
    86: "Ketoconazole"
})

counts = (
    subset_drug_ids.value_counts()
    .sort_index()
    .rename(index=drug_names)
)

print("\nNumber of drugs:", subset_drug_ids.nunique())
print("\nObservations per drug:")
print(counts)

assert counts.sum() == len(subset)

# Prepare the dataframe for outlier-analysis

review = subset.copy()

review["drug_id"] = subset_drug_ids
review["drug"] = subset_drug_ids.map(drug_names)

review["solvent_type"] = np.where(
    pure_solvent_mask,
    "pure",
    "binary"
)

# Create a grouping label from the recorded solvent IDs.
# Exclude N/A from the label, but retain any recorded zero-percent solvent.

def get_solvent_pair(text):
    ids = ast.literal_eval(text)

    return tuple(sorted(
        int(cid)
        for cid in ids
        if cid != "N/A"
    ))

review["solvent_pair"] = review["cmp_ids_solvent"].apply(
    get_solvent_pair
)

assert review["drug"].notna().all()

print("\nRecorded solvent systems per drug:")
print(review.groupby("drug")["solvent_pair"].nunique())

In [ ]:
# identifying the outliers using IQR method

iqr_review = review.copy()

iqr_review["LogS_iqr_flag"] = False
iqr_review["Temperature_iqr_flag"] = False

results = []

# Group by BOTH drug and solvent pair
for (drug, pair), group in iqr_review.groupby(
    ["drug", "solvent_pair"]
):
    # Screen each numerical variable separately
    for column, label in [
        ("value", "LogS"),
        ("Temperature, K", "Temperature")
    ]:
        values = group[column]

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        flags = (values < lower) | (values > upper)

        iqr_review.loc[group.index, f"{label}_iqr_flag"] = flags

        results.append({
            "Drug": drug,
            "Solvent pair": pair,
            "Variable": label,
            "Observations": len(values),
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower limit": lower,
            "Upper limit": upper,
            "Flagged": int(flags.sum())
        })

summary = pd.DataFrame(results)

print(summary.to_string(index=False))


flag_columns = ["LogS_iqr_flag", "Temperature_iqr_flag"]

any_flag = iqr_review[flag_columns].any(axis=1)

print("\nFlags per variable:")
print(iqr_review[flag_columns].sum())

print("\nUnique flagged observations:", any_flag.sum())

print(
    iqr_review.loc[
        any_flag,
        [
            "drug",
            "solvent_pair",
            "solvent_type",
            "value",
            "Temperature, K",
            "LogS_iqr_flag",
            "Temperature_iqr_flag"
        ]
    ].to_string()
)

In [ ]:
# Summarize IQR flags without printing every group

flag_columns = ["LogS_iqr_flag", "Temperature_iqr_flag"]

flagged_mask = iqr_review[flag_columns].any(axis=1)
flagged_rows = iqr_review.loc[flagged_mask].copy()

print("Total observations:", len(iqr_review))
print("LogS flags:", int(iqr_review["LogS_iqr_flag"].sum()))
print("Temperature flags:", int(iqr_review["Temperature_iqr_flag"].sum()))
print("Unique flagged observations:", len(flagged_rows))
print(f"Percentage flagged: {100 * flagged_mask.mean():.2f}%")

print("\nFlagged observations by solvent type:")
print(flagged_rows["solvent_type"].value_counts().to_string())

# Show only group-variable combinations with at least one flag
flagged_groups = summary.loc[summary["Flagged"] > 0]

print("\nGroup-variable combinations with flags:", len(flagged_groups))
display(flagged_groups.head(20))

# Preserve the complete results for inspection
summary.to_csv("iqr_summary_all_drugs.csv", index=False)

flagged_rows.to_csv(
    "iqr_flagged_rows_all_drugs.csv",
    index=True,
    index_label="original_row_index"
)

In [ ]:
# deciding whether to remove or retain the outlier

import matplotlib.pyplot as plt

plot_review = iqr_review.copy()

flag_columns = ["LogS_iqr_flag", "Temperature_iqr_flag"]

plot_review["any_iqr_flag"] = plot_review[flag_columns].any(axis=1)


def get_active_solvent_ids(row):
    ids = ast.literal_eval(row["cmp_ids_solvent"])
    fractions = ast.literal_eval(row["cmp_mole_fractions_solvent"])

    return tuple(sorted(
        int(cid)
        for cid, fraction in zip(ids, fractions)
        if fraction > 0
    ))


def get_first_solvent_fraction(row):
    ids = ast.literal_eval(row["cmp_ids_solvent"])
    fractions = ast.literal_eval(row["cmp_mole_fractions_solvent"])

    fraction_by_id = dict(zip(ids, fractions))

    # solvent_pair contains only recorded numeric IDs, excluding N/A
    first_id = row["solvent_pair"][0]

    return fraction_by_id[first_id]


plot_review["active_solvent_ids"] = plot_review.apply(
    get_active_solvent_ids,
    axis=1
)

plot_review["x_first"] = plot_review.apply(
    get_first_solvent_fraction,
    axis=1
)

# List every drug–solvent group containing flagged observations
review_queue = (
    plot_review.loc[plot_review["any_iqr_flag"]]
    .groupby(["drug", "solvent_pair"])
    .size()
    .reset_index(name="flagged_count")
)

review_queue.index.name = "group_number"

print("Number of flagged groups:", len(review_queue))
display(review_queue)


def plot_flagged_group(group_number):
    """Inspect one group using its number in review_queue."""

    if not 0 <= group_number < len(review_queue):
        raise ValueError(
            f"Choose a group number from 0 to {len(review_queue) - 1}"
        )

    selected = review_queue.iloc[group_number]

    drug = selected["drug"]
    pair = selected["solvent_pair"]

    same_drug = plot_review["drug"].eq(drug)

    fig, ax = plt.subplots(figsize=(10, 6))

    if len(pair) == 1:
        # Include all pure observations of this drug in this actual solvent,
        # even if some were recorded with a named zero-percent partner.
        same_active_solvent = plot_review[
            "active_solvent_ids"
        ].apply(lambda ids: ids == pair)

        plot_data = plot_review.loc[
            same_drug & same_active_solvent
        ].copy()

        plot_data = plot_data.sort_values("Temperature, K")

        x_column = "Temperature, K"

        ax.plot(
            plot_data[x_column],
            plot_data["value"],
            marker="o",
            markersize=4,
            linewidth=1,
            label="Pure-solvent observations"
        )

        ax.set_xlabel("Temperature (K)")
        ax.set_title(f"{drug} — pure solvent ID {pair[0]}")

    else:
        # Include mixtures and any endpoints recorded under this pair.
        same_pair = plot_review["solvent_pair"].apply(
            lambda ids: ids == pair
        )

        plot_data = plot_review.loc[
            same_drug & same_pair
        ].copy()

        x_column = "x_first"

        for temperature, series in plot_data.groupby("Temperature, K"):

            series = series.sort_values(x_column)

            ax.plot(
                series[x_column],
                series["value"],
                marker="o",
                markersize=4,
                linewidth=1,
                label=f"{temperature:g} K"
            )

        ax.set_xlabel(f"Mole fraction of solvent {pair[0]}")
        ax.set_xlim(-0.03, 1.03)
        ax.set_title(f"{drug} — solvent IDs {pair}")

    # Highlight and label every flagged observation shown
    flagged_points = plot_data.loc[
        plot_data["any_iqr_flag"]
    ]

    ax.scatter(
        flagged_points[x_column],
        flagged_points["value"],
        s=130,
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        zorder=5,
        label="IQR-flagged observations"
    )

    for idx, row in flagged_points.iterrows():
        ax.annotate(
            str(idx),
            (row[x_column], row["value"]),
            xytext=(5, 6),
            textcoords="offset points",
            fontsize=8
        )

    ax.set_ylabel("Log solubility (recorded value)")
    ax.grid(alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    fig.tight_layout()
    plt.show()

    print(f"Review group {group_number}: {drug}, {pair}")
    print("Flagged observations shown:")

    display(
        flagged_points[
            [
                "drug",
                "solvent_pair",
                "solvent_type",
                "value",
                "Temperature, K",
                "cmp_ids_solvent",
                "cmp_mole_fractions_solvent",
                "LogS_iqr_flag",
                "Temperature_iqr_flag"
            ]
        ]
    )

    return fig


In [ ]:
# Retain IQR-flagged observations by policy

flag_columns = ["LogS_iqr_flag", "Temperature_iqr_flag"]
flagged_mask = iqr_review[flag_columns].any(axis=1)

outlier_policy_report = iqr_review.loc[flagged_mask].copy()
outlier_policy_report.index.name = "original_row_index"

outlier_policy_report["decision"] = "retain"

outlier_policy_report["review_status"] = (
    "Individual review not documented"
)

outlier_policy_report["reason"] = (
    "Retained under the no-automatic-IQR-exclusion policy. "
    "An IQR flag alone does not establish a data error; "
    "individual validity has not been established."
)

outlier_policy_report["evidence"] = (
    "Dataset passed numeric, composition, and raw-data consistency checks. "
    "These checks do not establish the accuracy of the original measurement."
)

print("Flagged observations:", len(outlier_policy_report))
print("Default decision: retain")

In [ ]:
# Only enter errors supported by specific evidence.
# Keys must be original row indices of flagged observations.

confirmed_outlier_errors = {
    # Example structure:
    # ORIGINAL_ROW_INDEX: {
    #     "reason": "Describe the confirmed error and why it cannot be corrected.",
    #     "evidence": "Identify the source or check confirming the error."
    # }
}

unknown_indices = (
    set(confirmed_outlier_errors)
    - set(outlier_policy_report.index)
)

assert not unknown_indices, (
    f"These indices are not flagged observations: {unknown_indices}"
)

for idx, details in confirmed_outlier_errors.items():

    reason = details.get("reason", "")
    evidence = details.get("evidence", "")

    assert isinstance(reason, str) and reason.strip(), (
        f"Provide a reason for removing row {idx}"
    )

    assert isinstance(evidence, str) and evidence.strip(), (
        f"Provide supporting evidence for row {idx}"
    )

    outlier_policy_report.at[idx, "decision"] = "remove"
    outlier_policy_report.at[idx, "review_status"] = "Confirmed error"
    outlier_policy_report.at[idx, "reason"] = reason
    outlier_policy_report.at[idx, "evidence"] = evidence

print("Confirmed errors to remove:", len(confirmed_outlier_errors))

In [ ]:
# Remove only explicitly documented, confirmed errors

remove_indices = outlier_policy_report.index[
    outlier_policy_report["decision"].eq("remove")
]

final_data = cleaned.drop(index=remove_indices).copy()

# Verify the row count and absence of previously identified duplicates
assert len(final_data) == len(cleaned) - len(remove_indices)
assert not duplicate_keys.loc[final_data.index].duplicated().any()

# Save the original data columns, including all drugs and solvent types
final_data.to_csv(
    "final_cleaned_all_drugs_pure_and_binary.csv",
    index=False
)

# Save flags, policy decisions, review status, reasons, and evidence
outlier_policy_report.to_csv(
    "outlier_policy_report.csv",
    index=True,
    index_label="original_row_index"
)

retained_flags = outlier_policy_report["decision"].eq("retain").sum()

print("Rows before outlier decisions:", len(cleaned))
print("IQR-flagged observations retained:", int(retained_flags))
print("Confirmed erroneous observations removed:", len(remove_indices))
print("Final dataset rows:", len(final_data))
print("\nFinal dataset and outlier policy report saved.")